# CI com GitHub Actions — Bella Tavola 🍝
## Parte 1: Fundamentos de CI e GitHub Actions

# BLOCO 1

## Exercício 1.1 — Diagnóstico: o que pode quebrar?

In [1]:
# 1. Um campo de entrada pode ser renomeado sem atualizar os clientes da API.
# 2. A ordem das features do modelo pode mudar e gerar predições erradas sem erro explícito.
# 3. Um validador pode ser removido e a API passar a aceitar dados inválidos.
# 4. Um filtro pode parar de funcionar e retornar dados errados.
# 5. O modelo publicado pode ser trocado por outro incompatível com a API.
# 6. Uma refatoração pode gerar import circular e impedir a API de subir.

## Exercício 1.2 — CI vs CD na prática

In [2]:
# 1. A cada pull request, os testes de pytest rodam automaticamente.
#    Classificação: CI
#    Por quê: Verifica automaticamente se a integração do código novo está correta.

# 2. Após os testes passarem, um novo container Docker é gerado
#    e armazenado no registry, aguardando aprovação para ir a produção.
#    Classificação: CD — Entrega Contínua
#    Por quê: O artefato fica pronto para deploy, mas ainda depende de decisão humana.

# 3. Quando o branch main recebe um merge, a API é publicada
#    automaticamente no servidor de produção sem nenhuma aprovação manual.
#    Classificação: CD — Implantação Contínua
#    Por quê: O deploy acontece automaticamente, sem intervenção humana.

# 4. O pipeline verifica se o requirements.txt está atualizado
#    em relação às importações do código.
#    Classificação: CI
#    Por quê: É uma verificação de qualidade durante a integração.

# 5. Após o merge, o model.pkl mais recente é baixado do Hugging Face
#    Hub e os testes de predição são executados com ele.
#    Classificação: CI
#    Por quê: Está validando se o comportamento esperado continua correto.

# BLOCO 2

## Exercício 2.1 — Lendo um workflow

In [3]:
# 1. Em quais situações este workflow é disparado?
#    Ele roda em push para main, push para develop e pull request para main.

# 2. Quantos jobs existem? Eles rodam em paralelo ou em sequência?
#    Existem dois jobs: qualidade e relatorio.
#    Eles rodam em sequência, porque relatorio depende de qualidade com needs.

# 3. O job 'relatorio' sempre roda? Quando ele não roda?
#    Não. Ele não roda em pull requests nem em pushes para develop.
#    Ele só roda quando a referência é o branch main.

# 4. O que faz o --tb=short no comando do pytest?
#    Mostra um traceback resumido quando um teste falha.

# 5. O step "Verificar formatação" vai falhar se o código
#    não estiver formatado com black. Isso é desejável? Por quê?
#    Sim. Isso é desejável porque mantém o código padronizado e reduz ruído em revisões.

## Exercício 2.2 — Corrigindo um YAML inválido

In [4]:
# Erro 1: runs-on está fora da indentação correta do job test.
# Erro 2: python-version não está indentado dentro de with.
# Erro 3: o comando dentro de run: | não está indentado corretamente.
# Erro 4: a referência ao secret está errada; o correto é ${{ secrets.HF_TOKEN }}.

## Exercício 2.3 — Escrevendo o workflow do Bella Tavola

name: CI — Bella Tavola

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

jobs:
  verificar-api:
    runs-on: ubuntu-latest

    steps:
      - name: Baixar o código
        uses: actions/checkout@v4

      - name: Configurar Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"

      - name: Instalar dependências
        run: |
          python -m pip install --upgrade pip
          pip install -r aula_01_e02_fastapi/requirements.txt

      - name: Iniciar a API em background
        run: |
          cd aula_01_e02_fastapi
          python -m uvicorn main:app --host 0.0.0.0 --port 8000 &
          sleep 3

      - name: Verificar que a API responde
        run: |
          curl --fail http://localhost:8000/
          echo " API respondeu com sucesso"

## Exercício 2.4 — Desafio: múltiplos jobs 🎯

name: CI — Bella Tavola

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

jobs:
  verificar-sintaxe:
    runs-on: ubuntu-latest

    steps:
      - name: Baixar o código
        uses: actions/checkout@v4

      - name: Configurar Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"

      - name: Instalar ferramentas de qualidade
        run: |
          python -m pip install --upgrade pip
          pip install autoflake black

      - name: Verificar imports não utilizados
        run: autoflake --check --remove-all-unused-imports -r aula_01_e02_fastapi

      - name: Verificar formatação
        run: black --check aula_01_e02_fastapi

  verificar-api:
    runs-on: ubuntu-latest
    needs: verificar-sintaxe

    steps:
      - name: Baixar o código
        uses: actions/checkout@v4

      - name: Configurar Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"

      - name: Instalar dependências
        run: |
          python -m pip install --upgrade pip
          pip install -r aula_01_e02_fastapi/requirements.txt

      - name: Iniciar a API em background
        run: |
          cd aula_01_e02_fastapi
          python -m uvicorn main:app --host 0.0.0.0 --port 8000 &
          sleep 3

      - name: Verificar que a API responde
        run: |
          curl --fail http://localhost:8000/
          echo "API respondeu com sucesso"

In [6]:
# O que acontece na aba Actions quando verificar-sintaxe falha?
# O job verificar-sintaxe aparece com falha e o problema fica visível diretamente nos logs.

# O que acontece com o job verificar-api quando o anterior falha?
# O job verificar-api não é executado, porque depende do sucesso do job anterior com needs.